In [29]:
%load_ext sql
%reload_ext sql
%config SqlMagic.displaylimit = None
%sql sqlite:///../data/video_data.db
%config SqlMagic.feedback = 0

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


displaylimit: Value None will be treated as 0 (no limit)

displaylimit: Value None will be treated as 0 (no limit)

displaylimit: Value None will be treated as 0 (no limit)

displaylimit: Value None will be treated as 0 (no limit)

displaylimit: Value None will be treated as 0 (no limit)

displaylimit: Value None will be treated as 0 (no limit)

In [30]:
# read the data table
video_data = %sql SELECT * FROM videos;
df = video_data.DataFrame()

data_cols = ['id', 'snippet_publishedAt', 'snippet_channelId', 'snippet_title', 'snippet_description', 'snippet_categoryId', 'snippet_channelTitle', 'snippet_tags', 
             'statistics_viewCount', 'statistics_likeCount', 'statistics_commentCount', 'contentDetails_duration', 'contentDetails_definition', 'contentDetails_caption',
             'topicDetails_topicCategories', 'timestamp', 'query']
df = df[data_cols]

category_map = {1: 'Film & Animation', 2: 'Autos & Vehicles', 10: 'Music', 15: 'Pets & Animals', 17: 'Sports', 18: 'Short Movies', 19: 'Travel & Events', 20: 'Gaming',
                21: 'Videoblogging', 22: 'People & Blogs', 23: 'Comedy', 24: 'Entertainment', 25: 'News & Politics', 26: 'Howto & Style', 27: 'Education', 28: 'Science & Technology',
                29: 'Nonprofits & Activism', 30: 'Movies', 31: 'Anime/Animation', 32: 'Action/Adventure', 33: 'Classics', 34: 'Comedy', 35: 'Documentary', 36: 'Drama',
                37: 'Family', 38: 'Foreign', 39: 'Horror', 40: 'Sci-Fi/Fantasy', 41: 'Thriller', 42: 'Shorts', 43: 'Shows', 44: 'Trailers'}

df['snippet_categoryId'] = df['snippet_categoryId'].astype(int).map(category_map)

# adjust view counts to be channelId percentiled
df['statistics_viewCount'] = df['statistics_viewCount'].fillna(0).astype(int)
df['adjViewCount'] = df.groupby('snippet_channelId')['statistics_viewCount'].transform(lambda x: (x - x.min()) / (x.max() - x.min()))
df.sort_values('adjViewCount', ascending=False)
df.dropna(subset=['adjViewCount'], inplace=True)

df['snippet_title'] = df['snippet_title'].str.replace(r'[^\x00-\x7F]+', '', regex=True)
df['snippet_title'] = df['snippet_title'].str.replace(r'[^a-zA-Z0-9 ]', '', regex=True)
df['snippet_title'] = df['snippet_title'].str.lower()

c:\Users\Mitchell\Projects\Tools\ThumbnailModel\.conda\Lib\site-packages\sql\connection\connection.py:867: JupySQLRollbackPerformed: Found invalid transaction. JupySQL executed a ROLLBACK operation.
  warnings.warn(


In [ ]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

tokens = df['snippet_title'].apply(word_tokenize)

lemmatizer = WordNetLemmatizer()
lemmatized_tokens = tokens.apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
df['lemma_title'] = lemmatized_tokens.apply(lambda x: ' '.join(x))

X_train_lemma, X_test_lemma, y_train_lemma, y_test_lemma = train_test_split(df['lemma_title'], df['adjViewCount'], test_size=0.2, random_state=42)

In [ ]:
import numpy as np

min_features_size = int(np.sqrt(len(X_train_lemma)))
lemma_tfidf = TfidfVectorizer(stop_words='english', max_features=min_features_size)

X_train_lemma_tfidf = lemma_tfidf.fit_transform(X_train_lemma)
X_test_lemma_tfidf = lemma_tfidf.transform(X_test_lemma)

print(X_train_lemma_tfidf.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

from scipy.sparse import vstack

# Normalize sparse matrix by channel
def normalize_sparse_matrix(matrix, groups):
    unique_groups = np.unique(groups)
    normalized_matrices = []
    for group in unique_groups:
        group_indices = np.where(groups == group)[0]
        group_matrix = matrix[group_indices, :]
        scaler = StandardScaler(with_mean=False)  # with_mean=False because sparse matrices don't support centering
        group_matrix_normalized = scaler.fit_transform(group_matrix)
        normalized_matrices.append(group_matrix_normalized)
    return vstack(normalized_matrices)

channels_train = df.loc[X_train_lemma.index, 'snippet_channelId'].values
channels_test = df.loc[X_test_lemma.index, 'snippet_channelId'].values

X_train_lemma_tfidf_normalized = normalize_sparse_matrix(X_train_lemma_tfidf, channels_train)
X_test_lemma_tfidf_normalized = normalize_sparse_matrix(X_test_lemma_tfidf, channels_test)

# Output the shape to ensure everything is working
print(X_train_lemma_tfidf_normalized.shape)
print(X_test_lemma_tfidf_normalized.shape)

In [ ]:
# Continue with model training
from sklearn.linear_model import Lasso
from xgboost import XGBRegressor

# Lasso
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_lemma_tfidf_normalized, y_train_lemma)
print('Lasso', lasso.score(X_test_lemma_tfidf_normalized, y_test_lemma))

# XGBoost
xgb = XGBRegressor()
xgb.fit(X_train_lemma_tfidf_normalized, y_train_lemma)
print('XGBoost', xgb.score(X_test_lemma_tfidf_normalized, y_test_lemma))

In [ ]:
coef = pd.DataFrame({'feature': lemma_tfidf.get_feature_names_out(), 'coef': lasso.coef_}).sort_values('coef', ascending=False)
imps = pd.DataFrame({'feature': lemma_tfidf.get_feature_names_out(), 'importance': xgb.feature_importances_}).sort_values('importance', ascending=False)
features = pd.merge(coef, imps, on='feature')
features = features[features['coef'] * features['importance'] > 0]
features_list = features['feature'].tolist()
X_train_lemma_tfidf_normalized = X_train_lemma_tfidf_normalized[:, features.index]
X_test_lemma_tfidf_normalized = X_test_lemma_tfidf_normalized[:, features.index]

In [ ]:

plot_coef = features[features['coef'] > 0].sort_values('coef', ascending=False)
colors = plot_coef['importance']
colors = (colors - colors.min()) / (colors.max() - colors.min())

# plot the coefficients as barh
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 64))
plt.barh(plot_coef.index, plot_coef['coef'], color=plt.cm.viridis(colors))
plt.xlabel('Coefficient')
plt.ylabel('Feature')

plt.show()

In [ ]:
# creat poly features
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
X_train_lemma_tfidf_normalized_poly = poly.fit_transform(X_train_lemma_tfidf_normalized)
X_test_lemma_tfidf_normalized_poly = poly.transform(X_test_lemma_tfidf_normalized)

# train XGB, Lasso with selected features
lasso = Lasso(alpha=0.0001)
lasso.fit(X_train_lemma_tfidf_normalized_poly, y_train_lemma)
print('Lasso', lasso.score(X_test_lemma_tfidf_normalized_poly, y_test_lemma))

xgb = XGBRegressor()
xgb.fit(X_train_lemma_tfidf_normalized, y_train_lemma)
print('XGBoost', xgb.score(X_test_lemma_tfidf_normalized, y_test_lemma))

In [ ]:
coefs = pd.DataFrame({'feature': poly.get_feature_names_out(), 'coef': lasso.coef_}).sort_values('coef', ascending=False)
imps = pd.DataFrame({'feature': features_list, 'importance': xgb.feature_importances_}).sort_values('importance', ascending=False)
features = pd.merge(coefs, imps, on='feature')
features = features[features['coef'] * features['importance'] > 0]

In [ ]:

plot_coef = features[features['coefficient'] > 0].sort_values('coefficient', ascending=False)
colors = plot_coef['importance']
colors = (colors - colors.min()) / (colors.max() - colors.min())

# plot the coefficients as barh
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 64))
plt.barh(plot_coef.index, plot_coef['coefficient'], color=plt.cm.viridis(colors))
plt.xlabel('Coefficient')
plt.ylabel('Feature')

plt.show()

In [ ]:
pd.set_option('display.max_colwidth', 256)
pd.set_option('display.width', 512)

top_features = plot_coef.head(10).index.values
for fet in top_features:
    # get titles containing the feature
    titles = df[df['lemma_title'].str.contains(fet)]
    titles = titles.set_index('id')
    titles = titles.drop(columns=['contentDetails_definition', 'topicDetails_topicCategories', 'timestamp', 'query', 
                                  'adjViewCount', 'statistics_likeCount', 'statistics_commentCount', 'lemma_title', 'stem_title',
                                  'contentDetails_caption'])
    print(fet)
    print(titles)
    print(len(titles))
    print()
    print()
    print()